In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
import joblib

# 1. 加载并预处理数据
df = pd.read_excel(r'updated_fc_predictions_with_fc_hat_all_data.xlsx', sheet_name='7all')
df.dropna(inplace=True)

# 确保特征列表与你之前一致
features = ['PC', 'PC_TYPE','FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
            'VOID', 'w/b', 'b/a','SCM%', 'CAGG%', 'FAGG%', 'FA%','SS%','SF%']

X = df[features].values
Y = df['fc (MPa)'].values

# 2. 划分数据集 (由于 XGBoost 通常不需要标准化特征，这里直接使用原数据训练)
Xtrain, Xtest, ytrain, ytest = train_test_split(X, Y, train_size=0.85, random_state=42)

# 3. 使用最佳参数创建并训练 XGBoost 模型
best_params = {
    'subsample': 0.7, 
    'n_estimators': 300, 
    'max_depth': 4, 
    'learning_rate': 0.1, 
    'colsample_bytree': 0.7,
    'objective': 'reg:squarederror',
    'random_state': 42
}

xgb_model = XGBRegressor(**best_params)
xgb_model.fit(Xtrain, ytrain)

# 4. 准备 PDP 数据
# 我们要在原尺度上进行变化，这样更直观
mean_values = X.mean(axis=0)  # 计算原始特征的均值
wb_index = features.index('w/b')
# 生成 w/b 从最小值到最大值的 100 个点
wb_range = np.linspace(X[:, wb_index].min(), X[:, wb_index].max(), 100)

# 构建测试矩阵：其他列全为均值，只有 w/b 这一列在变
pdp_input = np.tile(mean_values, (wb_range.size, 1))
pdp_input[:, wb_index] = wb_range

# 5. 模型预测
pdp_predictions = xgb_model.predict(pdp_input)

# 6. 保存结果
pdp_results = pd.DataFrame({
    'w/b': wb_range, 
    'Predicted_fc': pdp_predictions
})

# 保存为 CSV
pdp_results.to_csv('pdp_XGB_wb_7.csv', index=False)

# 保存模型
joblib.dump(xgb_model, "XGB_model_best_7.joblib")

print("✨ XGBoost PDP 数据生成成功！")
print(f"结果已保存至: pdp_XGB_wb_7.csv")
print(f"模型已保存至: XGB_model_best_7.joblib")

✨ XGBoost PDP 数据生成成功！
结果已保存至: pdp_XGB_wb_7.csv
模型已保存至: XGB_model_best_7.joblib
